### Strctured Output 
- we can request the models to give the response in structured format or given schema
- Langchain supports multiple schemas
- The Librarys we used for defining the schema are pydantic,typeDict,basemodels

### Pydantic
- Pydantic is good for field validation and it supports nested strctures also
- It provides runtime validation

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain.chat_models import init_chat_model
model=init_chat_model(
  'gpt-6-luna'
)
model

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.11', 'langchain': '1.4.6', 'langchain-openai': '1.7.2'}}, profile={'name': 'GPT-6 Luna', 'release_date': '2026-09-22', 'last_updated': '2026-09-22', 'open_weights': False, 'max_input_tokens': 1050000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'reasoning_effort_levels': ['none', 'low', 'medium', 'high', 'xhigh', 'max'], 'reasoning_effort_default': 'medium', 'mid_conversation_system_messages': True, 'mid_conversation_tools': False}, client=<openai.resources.chat.completions.completions.Completions ob

In [6]:
## import the Field and BaseModel from pydantic
from pydantic import Field,BaseModel

## define the output schema using pydantic
class Movie(BaseModel):
  title:str=Field(description="The title of the movie")
  director:str=Field(description="The director of the movie")
  year:int=Field(description="Release data of the movie")
  rating:float=Field(description="Rating of this movie")

## define the model
model_with_structure=model.with_structured_output(Movie)

In [7]:
## invoke the model
model_with_structure.invoke("Provide the details about avengers")

Movie(title='The Avengers', director='Joss Whedon', year=2012, rating=8.0)

## Messaged output along side parsed structure

In [8]:
## import the Field and BaseModel from pydantic
from pydantic import Field,BaseModel

## define the output schema using pydantic
class Movie(BaseModel):
  title:str=Field(...,description="The title of the movie")
  director:str=Field(...,description="The director of the movie")
  year:int=Field(...,description="Release data of the movie")
  rating:float=Field(...,description="Rating of this movie")

## define the model
model_with_structure=model.with_structured_output(Movie,include_raw=True)

## invoke the model
model_with_structure.invoke("Provide the details about avengers")

{'raw': AIMessage(content='{"title":"The Avengers","director":"Joss Whedon","year":2012,"rating":8.0}', additional_kwargs={'parsed': Movie(title='The Avengers', director='Joss Whedon', year=2012, rating=8.0), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 122, 'prompt_tokens': 109, 'total_tokens': 231, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 85, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EdwB6H8mdemcVULej3awsR96WsdiA', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a18248-40fc-7aa1-adb2-00d48a759336-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 109, 'output_tokens': 122, 'total_tokens': 231,

## Pydantic with Nested Structure 

In [13]:
from pydantic import Field,BaseModel

class Actor(BaseModel):
  name:str=Field(description='Tell me actor name')
  role:str=Field(description='Tell me the role of the actor in this movie')

class Movie(BaseModel):
  name:str
  year:int
  cast:list[Actor]
  generes:list[str]
  budget:float | None=Field(default=None,description='Budget in USD')

model_with_structure=model.with_structured_output(Movie)
model_with_structure.invoke("Provide the details about avengers")

Movie(name='The Avengers', year=2012, cast=[Actor(name='Robert Downey Jr.', role='Tony Stark / Iron Man'), Actor(name='Chris Evans', role='Steve Rogers / Captain America'), Actor(name='Mark Ruffalo', role='Bruce Banner / Hulk'), Actor(name='Chris Hemsworth', role='Thor'), Actor(name='Scarlett Johansson', role='Natasha Romanoff / Black Widow'), Actor(name='Jeremy Renner', role='Clint Barton / Hawkeye'), Actor(name='Tom Hiddleston', role='Loki'), Actor(name='Samuel L. Jackson', role='Nick Fury')], generes=['Superhero', 'Action', 'Science fiction', 'Adventure'], budget=220000000.0)

## TypeDict 
- TypeDict is used for to describe the shape of the dictionary 
- And if you dont want run time validation

In [ ]:
# create a schema using TypeDict
from typing_extensions import TypedDict,Annotated

class Movie(TypedDict):
  """Get the Movie details"""
  name:Annotated[str,"The title of the movie"]
  actor:Annotated[str,"The name of the actor"]
  rating:Annotated[float,"rating of the movie"]
  budget:Annotated[int,"Give the budget of the movie in USD"]

model_with_typedDict=model.with_structured_output(Movie)
model_with_typedDict.invoke("Give me the movie inception details")


{'name': 'Inception',
 'actor': 'Leonardo DiCaprio',
 'rating': 8.8,
 'budget': 160000000}

In [24]:
## create the agent + Pydantic
from langchain.agents import create_agent
from pydantic import Field,BaseModel

# define the schema using pydantic
class movie_details(BaseModel):
  """ Give the Proper movie details"""
  movie_name:str=Field(description='The name of the movie')
  actor:str=Field(description='The name of the actor in this movie')
  rating:float=Field(decription='The rating of the movie ')
  collections:int=Field(description='The collections of the movie in cr')

# create agent
agent_pydantic=create_agent(
  model='gpt-6-luna',
  response_format=movie_details
)
result=agent_pydantic.invoke({
  "messages":[
    {
      'role':'user',
      'content':'Give me the telugu movie OG details'
    }
  ]
})
result['structured_response']


C:\Users\saika\AppData\Local\Temp\ipykernel_30084\2672915113.py:10: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'decription'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  rating:float=Field(decription='The rating of the movie ')


movie_details(movie_name='They Call Him OG (OG)', actor='Pawan Kalyan', rating=6.2, collections=300)